In [4]:
import pandas as pd
from pathlib import Path

rain_file = Path("../data/processed/kelani/kelani_daily_rainfall_combined.xlsx")
wl_file = Path("../data/processed/kelani/kelani_hourly_waterlevel_combined.xlsx")

# Load rainfall workbook
rain_sheets = pd.read_excel(rain_file, sheet_name=None, engine="openpyxl")
rain_df = pd.concat(rain_sheets.values(), ignore_index=True)

# Load water-level workbook
wl_sheets = pd.read_excel(wl_file, sheet_name=None, engine="openpyxl")
wl_df = pd.concat(wl_sheets.values(), ignore_index=True)

# Clean column names
rain_df.columns = rain_df.columns.astype(str).str.strip()
wl_df.columns = wl_df.columns.astype(str).str.strip()

# Water-level file already has a complete Date column
wl_df["Date"] = pd.to_datetime(wl_df["Date"], errors="coerce")

# Standardise station names for later merging
station_mapping = {
    "kitulgala": "Kithulgala"
}

rain_df["Location"] = rain_df["Location"].astype(str).str.strip()
wl_df["Location"] = wl_df["Location"].astype(str).str.strip()

rain_df["Location"] = rain_df["Location"].str.lower().map(
    lambda x: station_mapping.get(x, x.title())
)

wl_df["Location"] = wl_df["Location"].str.title()

# Inspect both datasets
print("Rainfall shape:", rain_df.shape)
print("Rainfall columns:", rain_df.columns.tolist())
print("\nWater-level shape:", wl_df.shape)
print("Water-level columns:", wl_df.columns.tolist())

display(wl_df.head())
display(rain_df.head())

Rainfall shape: (792, 36)
Rainfall columns: ['Location', 'YEARMONTH', 'YEAR', 'MONTH', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D21', 'D22', 'D23', 'D24', 'D25', 'D26', 'D27', 'D28', 'D29', 'D30', 'D31', 'TOTAL']

Water-level shape: (37978, 5)
Water-level columns: ['Location', 'Year', 'Month', 'Date', 'Avg. WL']


,Location,Year,Month,Date,Avg. WL
0,Norwood,1996,1,1996-01-01,1098.211667
1,Norwood,1996,1,1996-01-02,1098.253333
2,Norwood,1996,1,1996-01-03,1098.202917
3,Norwood,1996,1,1996-01-04,1098.143333
4,Norwood,1996,1,1996-01-05,1098.112500


,Location,YEARMONTH,YEAR,MONTH,D1,D2,D3,D4,D5,D6,...,D23,D24,D25,D26,D27,D28,D29,D30,D31,TOTAL
0,Norwood,14/15/04,2015,JAN,0.0,0.0,0.0,0.0,0.0,1.7,...,0.0,0.0,0.0,0.0,0.0,11.7,23.4,4.6,0.5,41.9
1,Norwood,14/15/05,2015,FEB,0.0,0.0,10.4,0.0,25.9,23.9,...,5.0,7.2,0.0,12.5,20.0,31.7,NaN,NaN,NaN,182.2
2,Norwood,14/15/06,2015,MAR,3.7,0.0,0.0,20.3,2.5,0.0,...,0.0,0.0,0.0,25.4,1.3,1.8,4.6,7.0,5.5,85.7
3,Norwood,14/15/07,2015,APR,39.0,26.8,0.7,13.1,0.0,0.3,...,12.5,39.3,21.4,1.2,38.8,1.2,3.1,8.7,NaN,341.4
4,Norwood,14/15/08,2015,MAY,1.9,0.0,0.0,3.6,29.9,9.7,...,1.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.1,125.2


In [5]:
print(rain_df.columns.tolist())
print(rain_df.head(3))
print(wl_df.columns.tolist())


['Location', 'YEARMONTH', 'YEAR', 'MONTH', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D21', 'D22', 'D23', 'D24', 'D25', 'D26', 'D27', 'D28', 'D29', 'D30', 'D31', 'TOTAL']
  Location YEARMONTH  YEAR MONTH   D1   D2    D3    D4    D5    D6  ...  D23  \
0  Norwood  14/15/04  2015   JAN  0.0  0.0   0.0   0.0   0.0   1.7  ...  0.0   
1  Norwood  14/15/05  2015   FEB  0.0  0.0  10.4   0.0  25.9  23.9  ...  5.0   
2  Norwood  14/15/06  2015   MAR  3.7  0.0   0.0  20.3   2.5   0.0  ...  0.0   

   D24  D25   D26   D27   D28   D29  D30  D31  TOTAL  
0  0.0  0.0   0.0   0.0  11.7  23.4  4.6  0.5   41.9  
1  7.2  0.0  12.5  20.0  31.7   NaN  NaN  NaN  182.2  
2  0.0  0.0  25.4   1.3   1.8   4.6  7.0  5.5   85.7  

[3 rows x 36 columns]
['Location', 'Year', 'Month', 'Date', 'Avg. WL']


In [6]:
# Convert monthly-wide rainfall into daily records

rain_df.columns = rain_df.columns.astype(str).str.strip()
rain_df["MONTH"] = rain_df["MONTH"].astype(str).str.upper().str.strip()

month_mapping = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

rain_df["MONTH_NUM"] = rain_df["MONTH"].map(month_mapping)

day_cols = [
    col for col in rain_df.columns
    if col.startswith("D") and col[1:].isdigit()
]

rain_daily = rain_df.melt(
    id_vars=["Location", "YEAR", "MONTH_NUM"],
    value_vars=day_cols,
    var_name="Day",
    value_name="Rainfall"
)

rain_daily["Day"] = rain_daily["Day"].str[1:].astype(int)

rain_daily["Date"] = pd.to_datetime(
    {
        "year": rain_daily["YEAR"],
        "month": rain_daily["MONTH_NUM"],
        "day": rain_daily["Day"]
    },
    errors="coerce"
)

# Remove invalid calendar dates, such as 31 February
rain_daily = rain_daily.dropna(subset=["Date"]).copy()

rain_daily["Rainfall"] = pd.to_numeric(
    rain_daily["Rainfall"], errors="coerce"
)

# Standardise station names for merging
rain_daily["Location"] = (
    rain_daily["Location"].astype(str).str.strip().str.title()
)

rain_daily["Location"] = rain_daily["Location"].replace({
    "Kitulgala": "Kithulgala"
})

rain_daily = rain_daily[["Location", "Date", "Rainfall"]]

print("Daily rainfall shape:", rain_daily.shape)
print("Date range:", rain_daily["Date"].min(), "to", rain_daily["Date"].max())
print("Stations:", sorted(rain_daily["Location"].unique()))

display(rain_daily.head(10))

Daily rainfall shape: (24108, 3)
Date range: 2015-01-01 00:00:00 to 2025-12-31 00:00:00
Stations: ['Deraniyagala', 'Glencourse', 'Hanwella', 'Holombuwa', 'Kithulgala', 'Norwood']


,Location,Date,Rainfall
0,Norwood,2015-01-01,0.0
1,Norwood,2015-02-01,0.0
2,Norwood,2015-03-01,3.7
3,Norwood,2015-04-01,39.0
4,Norwood,2015-05-01,1.9
5,Norwood,2015-06-01,0.0
6,Norwood,2015-07-01,0.0
7,Norwood,2015-08-01,0.0
8,Norwood,2015-09-01,0.0
9,Norwood,2015-10-01,0.0


In [7]:
# Prepare daily water-level data

wl_df.columns = wl_df.columns.astype(str).str.strip()
wl_df["Date"] = pd.to_datetime(wl_df["Date"], errors="coerce")
wl_df["Avg. WL"] = pd.to_numeric(wl_df["Avg. WL"], errors="coerce")

wl_daily = wl_df.rename(
    columns={"Avg. WL": "Water_Level"}
)[["Location", "Date", "Water_Level"]].copy()

wl_daily["Location"] = (
    wl_daily["Location"].astype(str).str.strip().str.title()
)

wl_daily["Location"] = wl_daily["Location"].replace({
    "Kitulgala": "Kithulgala"
})

wl_daily = wl_daily.dropna(
    subset=["Date", "Water_Level"]
)

# Check for duplicate station-date records before merging
print(
    "Duplicate water-level station/date rows:",
    wl_daily.duplicated(["Location", "Date"]).sum()
)

print(
    "Duplicate rainfall station/date rows:",
    rain_daily.duplicated(["Location", "Date"]).sum()
)

Duplicate water-level station/date rows: 0
Duplicate rainfall station/date rows: 0


In [8]:
# Merge only matching station-date records
df = pd.merge(
    wl_daily,
    rain_daily,
    on=["Location", "Date"],
    how="inner",
    validate="one_to_one"
)

print("Merged shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Stations:", sorted(df["Location"].unique()))

display(df.head(10))

Merged shape: (24108, 4)
Date range: 2015-01-01 00:00:00 to 2025-12-31 00:00:00
Stations: ['Deraniyagala', 'Glencourse', 'Hanwella', 'Holombuwa', 'Kithulgala', 'Norwood']


,Location,Date,Water_Level,Rainfall
0,Norwood,2015-01-01,1098.668333,0.0
1,Norwood,2015-01-02,1098.593750,0.0
2,Norwood,2015-01-03,1098.617917,0.0
3,Norwood,2015-01-04,1098.604583,0.0
4,Norwood,2015-01-05,1098.589583,0.0
5,Norwood,2015-01-06,1098.580000,1.7
6,Norwood,2015-01-07,1098.556250,0.0
7,Norwood,2015-01-08,1098.526667,0.0
8,Norwood,2015-01-09,1098.506250,0.0
9,Norwood,2015-01-10,1098.488750,0.0


In [9]:
import numpy as np
import pandas as pd

# Check that the merged dataset contains the required columns
required = {"Location", "Date", "Water_Level", "Rainfall"}
missing = required - set(df.columns)

if missing:
    raise ValueError(f"Missing columns in df: {missing}")

model_df = df.copy()
model_df["Date"] = pd.to_datetime(model_df["Date"], errors="coerce")
model_df["Water_Level"] = pd.to_numeric(model_df["Water_Level"], errors="coerce")
model_df["Rainfall"] = pd.to_numeric(model_df["Rainfall"], errors="coerce")

model_df = model_df.dropna(
    subset=["Location", "Date", "Water_Level", "Rainfall"]
).copy()

model_df = model_df.sort_values(["Location", "Date"])

# Rainfall history and water-level changes for each station
grouped = model_df.groupby("Location", group_keys=False)

model_df["Rainfall_3day"] = grouped["Rainfall"].transform(
    lambda s: s.rolling(3, min_periods=1).sum()
)
model_df["Rainfall_7day"] = grouped["Rainfall"].transform(
    lambda s: s.rolling(7, min_periods=1).sum()
)
model_df["Water_Level_Change"] = grouped["Water_Level"].diff().fillna(0)

# Relative historical ranks make stations with different baseline levels comparable.
model_df["WL_Rank"] = grouped["Water_Level"].rank(pct=True)
model_df["Rain_Rank"] = grouped["Rainfall_7day"].rank(pct=True)

# Exploratory proxy score: water level 70%, accumulated rainfall 30%
model_df["Severity_Score"] = (
    0.70 * model_df["WL_Rank"] +
    0.30 * model_df["Rain_Rank"]
)

# Three proxy classes
model_df["Severity"] = pd.cut(
    model_df["Severity_Score"],
    bins=[-np.inf, 0.50, 0.80, np.inf],
    labels=["Low", "Medium", "High"]
).astype(str)

print("Rows:", len(model_df))
print("Date range:", model_df["Date"].min(), "to", model_df["Date"].max())
print("\nClass counts:")
print(model_df["Severity"].value_counts())

Rows: 24103
Date range: 2015-01-01 00:00:00 to 2025-12-31 00:00:00

Class counts:
Severity
Low       12381
Medium     7474
High       4248
Name: count, dtype: int64


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

features = [
    "Location",
    "Water_Level",
    "Rainfall",
    "Rainfall_3day",
    "Rainfall_7day",
    "Water_Level_Change"
]

model_df = model_df.sort_values("Date").reset_index(drop=True)

# Split by date, not by randomly selected rows
dates = np.sort(model_df["Date"].unique())
if len(dates) < 5:
    raise ValueError("Not enough distinct dates to create a chronological train/test split.")

split_date = dates[int(len(dates) * 0.80)]

train = model_df[model_df["Date"] < split_date].copy()
test = model_df[model_df["Date"] >= split_date].copy()

if train["Severity"].nunique() < 2:
    raise ValueError("Training data has fewer than two severity classes.")

X_train = train[features]
y_train = train["Severity"]
X_test = test[features]
y_test = test["Severity"]

preprocessor = ColumnTransformer(
    transformers=[
        ("station", OneHotEncoder(handle_unknown="ignore"), ["Location"])
    ],
    remainder="passthrough"
)

flood_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=3,
        n_jobs=-1
    ))
])

flood_model.fit(X_train, y_train)

predictions = flood_model.predict(X_test)

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Test begins:", split_date)
print("\nAccuracy against proxy labels:", round(accuracy_score(y_test, predictions), 3))
print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    labels=["Low", "Medium", "High"],
    zero_division=0
))
print("Confusion matrix (Low, Medium, High):")
print(confusion_matrix(y_test, predictions, labels=["Low", "Medium", "High"]))

Training rows: 19281
Testing rows: 4822
Test begins: 2023-10-20T00:00:00.000000

Accuracy against proxy labels: 0.966

Classification report:
              precision    recall  f1-score   support

         Low       0.99      0.97      0.98      2142
      Medium       0.93      0.97      0.95      1687
        High       0.98      0.96      0.97       993

    accuracy                           0.97      4822
   macro avg       0.97      0.96      0.96      4822
weighted avg       0.97      0.97      0.97      4822

Confusion matrix (Low, Medium, High):
[[2070   72    0]
 [  29 1637   21]
 [   0   44  949]]


In [ ]:
def assess_flood_severity(
    station,
    water_level,
    rainfall,
    rainfall_3day,
    rainfall_7day,
    water_level_change
):
    """Return a prototype severity class for one station and date."""

    observation = pd.DataFrame([{
        "Location": station,
        "Water_Level": float(water_level),
        "Rainfall": float(rainfall),
        "Rainfall_3day": float(rainfall_3day),
        "Rainfall_7day": float(rainfall_7day),
        "Water_Level_Change": float(water_level_change)
    }])

    severity = flood_model.predict(observation[features])[0]

    return {
        "station": station,
        "severity": severity,
        "model_type": "Random Forest prototype",
        "label_status": "Exploratory proxy; not officially validated"
    }

In [12]:
latest = (
    model_df.sort_values("Date")
    .groupby("Location")
    .tail(1)
    .sort_values("Date")
)

for _, row in latest.iterrows():
    result = assess_flood_severity(
        station=row["Location"],
        water_level=row["Water_Level"],
        rainfall=row["Rainfall"],
        rainfall_3day=row["Rainfall_3day"],
        rainfall_7day=row["Rainfall_7day"],
        water_level_change=row["Water_Level_Change"]
    )
    print(result)

{'station': 'Kithulgala', 'severity': 'Low', 'model_type': 'Random Forest prototype', 'label_status': 'Exploratory proxy; not officially validated'}
{'station': 'Holombuwa', 'severity': 'Low', 'model_type': 'Random Forest prototype', 'label_status': 'Exploratory proxy; not officially validated'}
{'station': 'Hanwella', 'severity': 'Low', 'model_type': 'Random Forest prototype', 'label_status': 'Exploratory proxy; not officially validated'}
{'station': 'Glencourse', 'severity': 'Low', 'model_type': 'Random Forest prototype', 'label_status': 'Exploratory proxy; not officially validated'}
{'station': 'Deraniyagala', 'severity': 'Low', 'model_type': 'Random Forest prototype', 'label_status': 'Exploratory proxy; not officially validated'}
{'station': 'Norwood', 'severity': 'Low', 'model_type': 'Random Forest prototype', 'label_status': 'Exploratory proxy; not officially validated'}


In [13]:
import joblib
from pathlib import Path

output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(flood_model, output_dir / "kelani_flood_severity_rf.joblib")

print("Model saved to:", output_dir / "kelani_flood_severity_rf.joblib")

Model saved to: ../models/kelani_flood_severity_rf.joblib


In [14]:
import joblib
from pathlib import Path

model_path = Path("../models/kelani_flood_severity_rf.joblib")

loaded_model = joblib.load(model_path)

print("Model loaded successfully!")
print("Model classes:", loaded_model.named_steps["classifier"].classes_)

Model loaded successfully!
Model classes: ['High' 'Low' 'Medium']


In [15]:
print("Actual proxy-label distribution:")
print(model_df["Severity"].value_counts())

print("\nLatest observations by station:")
display(
    model_df.sort_values("Date")
    .groupby("Location")
    .tail(1)[[
        "Date",
        "Location",
        "Water_Level",
        "Rainfall",
        "Rainfall_7day",
        "Severity"
    ]]
)

Actual proxy-label distribution:
Severity
Low       12381
Medium     7474
High       4248
Name: count, dtype: int64

Latest observations by station:


,Date,Location,Water_Level,Rainfall,Rainfall_7day,Severity
24101,2025-12-31,Kithulgala,53.670917,40.7,45.7,Low
24097,2025-12-31,Holombuwa,45.019964,16.0,16.0,Low
24098,2025-12-31,Hanwella,1.064583,0.0,0.0,Low
24099,2025-12-31,Glencourse,8.607417,16.9,16.9,Low
24100,2025-12-31,Deraniyagala,73.524804,26.5,27.2,Low
24102,2025-12-31,Norwood,1098.346667,0.0,0.0,Low


In [16]:
latest = (
    model_df.sort_values("Date")
    .groupby("Location")
    .tail(1)
    .copy()
)

latest["Predicted_Severity"] = loaded_model.predict(latest[features])

results = latest[[
    "Date",
    "Location",
    "Water_Level",
    "Rainfall",
    "Rainfall_7day",
    "Predicted_Severity"
]]

output_path = Path("../data/processed/kelani/latest_severity_assessment.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

results.to_csv(output_path, index=False)

display(results)
print("Results saved to:", output_path)

,Date,Location,Water_Level,Rainfall,Rainfall_7day,Predicted_Severity
24101,2025-12-31,Kithulgala,53.670917,40.7,45.7,Low
24097,2025-12-31,Holombuwa,45.019964,16.0,16.0,Low
24098,2025-12-31,Hanwella,1.064583,0.0,0.0,Low
24099,2025-12-31,Glencourse,8.607417,16.9,16.9,Low
24100,2025-12-31,Deraniyagala,73.524804,26.5,27.2,Low
24102,2025-12-31,Norwood,1098.346667,0.0,0.0,Low


Results saved to: ../data/processed/kelani/latest_severity_assessment.csv
